# IEEE-CIS Fraud Detection — Modelling

## Goal

Train three gradient boosting models on the engineered features from notebook 02.
Each model is logged as a separate MLflow run so results are comparable.

## Why these three models?

| Model | Strength |
|---|---|
| XGBoost | Battle-tested, great default performance, wide community support |
| LightGBM | Fastest training, handles high cardinality well, good on large datasets |
| CatBoost | Best native categorical handling, less hyperparameter tuning needed |

All three are gradient boosting — ensemble of decision trees built sequentially,
each tree correcting the errors of the previous one.

## Evaluation Metric

**Primary: AUC-PR (Area Under Precision-Recall Curve)**

Not accuracy. Not AUC-ROC. AUC-PR because:
- 96.5% accuracy is achievable by predicting everything as legitimate — useless
- AUC-ROC is optimistic under class imbalance (28:1 ratio)
- AUC-PR focuses on the positive (fraud) class — more informative when fraud is rare

**Secondary: AUC-ROC, F1** — for completeness and comparison

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import json
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    f1_score,
    confusion_matrix,
    classification_report
)

os.chdir('/Users/shaliqshukoor/fraud-risk-system')

# Load processed data
train_df = pd.read_parquet('data/processed/train_features.parquet')
val_df   = pd.read_parquet('data/processed/val_features.parquet')

with open('data/processed/feature_names.json') as f:
    feature_cols = json.load(f)

X_train = train_df[feature_cols]
y_train = train_df['isFraud']
X_val   = val_df[feature_cols]
y_val   = val_df['isFraud']

print(f'X_train: {X_train.shape}')
print(f'X_val:   {X_val.shape}')
print(f'Features: {len(feature_cols)}')
print(f'Train fraud rate: {y_train.mean()*100:.2f}%')
print(f'Val fraud rate:   {y_val.mean()*100:.2f}%')

/Users/shaliqshukoor/fraud-risk-system/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


X_train: (442905, 443)
X_val:   (147635, 443)
Features: 443
Train fraud rate: 3.51%
Val fraud rate:   3.45%


In [2]:
from sklearn.preprocessing import LabelEncoder

# Label encode remaining object columns — fit on train only to prevent leakage
obj_cols = X_train.select_dtypes(include='object').columns.tolist()
print(f'Object columns to encode: {obj_cols}')

for col in obj_cols:
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col].astype(str))
    X_val[col]   = le.transform(X_val[col].astype(str).map(
        lambda x: x if x in le.classes_ else le.classes_[0]
    ))

print('Done. All columns are now numeric.')
print(X_train.dtypes.value_counts())

Object columns to encode: ['P_emaildomain', 'R_emaildomain', 'M4', 'id_12', 'id_15', 'id_16', 'id_23', 'id_27', 'id_28', 'id_29', 'id_30', 'id_31', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo', 'card4_6']
Done. All columns are now numeric.
float64    403
int64       40
Name: count, dtype: int64


## Chapter 1 — Class Imbalance Strategy

28:1 ratio means the model will see 28 legitimate transactions for every 1 fraud.
Without correction it will learn to predict everything as legitimate.

**Solution: `scale_pos_weight`**

Tells the model to penalise missed fraud more heavily by upweighting the positive class:

```
scale_pos_weight = count(negative) / count(positive)
                 = 569,877 / 20,663
                 ≈ 27.6
```

This makes the model treat each fraud case as if it were 27.6 legitimate cases —
balancing the gradient updates during training.

In [3]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f'scale_pos_weight: {scale_pos_weight:.2f}')

scale_pos_weight: 27.46


## Chapter 2 — MLflow Setup

Each model is logged as a separate MLflow run under one experiment.
This lets you compare AUC-PR, AUC-ROC, F1 side by side in the MLflow UI.

We log:
- All hyperparameters
- AUC-PR, AUC-ROC, F1 scores
- Confusion matrix as an artifact
- Feature importance plot as an artifact
- The model itself (for later deployment)

In [4]:
mlflow.set_experiment('fraud-detection')
print('MLflow experiment set: fraud-detection')

MLflow experiment set: fraud-detection


## Chapter 3 — XGBoost

Start with XGBoost — most familiar, good baseline.

Key parameters:
- `scale_pos_weight` — handles class imbalance
- `eval_metric='aucpr'` — optimises for AUC-PR during training
- `early_stopping_rounds=50` — stops if val AUC-PR doesn't improve for 50 rounds
- `n_estimators=1000` — high ceiling, early stopping will find the right number

### Your task

Fill in the blanks and run the XGBoost training cell below.

In [5]:
import joblib, os
from xgboost import XGBClassifier

os.makedirs('models', exist_ok=True)

xgb_params = {
    'n_estimators': 1000,
    'max_depth': 6,
    'learning_rate': 0.05,
    'scale_pos_weight': scale_pos_weight,
    'eval_metric': 'aucpr',
    'early_stopping_rounds': 50,
    'random_state': 42,
    'n_jobs': -1
}

with mlflow.start_run(run_name='xgboost_baseline'):
    mlflow.log_params(xgb_params)
    
    xgb_model = XGBClassifier(**xgb_params)
    xgb_model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=100
    )
    
    y_pred_proba = xgb_model.predict_proba(X_val)[:, 1]
    y_pred       = (y_pred_proba >= 0.5).astype(int)
    
    auc_pr  = average_precision_score(y_val, y_pred_proba)
    auc_roc = roc_auc_score(y_val, y_pred_proba)
    f1      = f1_score(y_val, y_pred)
    
    mlflow.log_metric('auc_pr',  auc_pr)
    mlflow.log_metric('auc_roc', auc_roc)
    mlflow.log_metric('f1',      f1)
    
    joblib.dump(xgb_model, 'models/xgboost_baseline.pkl')
    
    print(f'XGBoost Results:')
    print(f'  AUC-PR:  {auc_pr:.4f}')
    print(f'  AUC-ROC: {auc_roc:.4f}')
    print(f'  F1:      {f1:.4f}')
    print(f'  Best iteration: {xgb_model.best_iteration}')

[0]	validation_0-aucpr:0.30387
[100]	validation_0-aucpr:0.45620
[200]	validation_0-aucpr:0.47929
[300]	validation_0-aucpr:0.49308
[400]	validation_0-aucpr:0.50097
[500]	validation_0-aucpr:0.50755
[600]	validation_0-aucpr:0.51207
[700]	validation_0-aucpr:0.51463
[800]	validation_0-aucpr:0.51613
[900]	validation_0-aucpr:0.51846
[972]	validation_0-aucpr:0.51854
XGBoost Results:
  AUC-PR:  0.5190
  AUC-ROC: 0.8978
  F1:      0.4103
  Best iteration: 922
🏃 View run xgboost_baseline at: https://germanywestcentral.api.azureml.ms/mlflow/v2.0/subscriptions/1462cf45-0a1d-4624-a94a-ce9949a6ffd4/resourceGroups/shaliq_study/providers/Microsoft.MachineLearningServices/workspaces/shaliq_demo/#/experiments/51a2010c-aa0f-4cbe-a06b-ae4cdfbda8ca/runs/b20ecdf3-d344-42a8-a10d-d4fd4dffa3d2
🧪 View experiment at: https://germanywestcentral.api.azureml.ms/mlflow/v2.0/subscriptions/1462cf45-0a1d-4624-a94a-ce9949a6ffd4/resourceGroups/shaliq_study/providers/Microsoft.MachineLearningServices/workspaces/shaliq_demo

## Chapter 4 — LightGBM

LightGBM is faster than XGBoost on large datasets — uses leaf-wise tree growth
instead of level-wise, finding better splits faster.

Key difference from XGBoost:
- `is_unbalance=True` instead of `scale_pos_weight` — automatic balancing
- `metric='average_precision'` — equivalent to AUC-PR

In [6]:
from lightgbm import LGBMClassifier, early_stopping, log_evaluation

lgbm_params = {
    'n_estimators': 1000,
    'max_depth': 6,
    'learning_rate': 0.05,
    'is_unbalance': True,
    'metric': 'average_precision',
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1
}

with mlflow.start_run(run_name='lightgbm_baseline'):
    mlflow.log_params(lgbm_params)
    
    lgbm_model = LGBMClassifier(**lgbm_params)
    lgbm_model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[early_stopping(50), log_evaluation(100)]
    )
    
    y_pred_proba = lgbm_model.predict_proba(X_val)[:, 1]
    y_pred       = (y_pred_proba >= 0.5).astype(int)
    
    auc_pr  = average_precision_score(y_val, y_pred_proba)
    auc_roc = roc_auc_score(y_val, y_pred_proba)
    f1      = f1_score(y_val, y_pred)
    
    mlflow.log_metric('auc_pr',  auc_pr)
    mlflow.log_metric('auc_roc', auc_roc)
    mlflow.log_metric('f1',      f1)
    
    joblib.dump(lgbm_model, 'models/lightgbm_baseline.pkl')
    
    print(f'LightGBM Results:')
    print(f'  AUC-PR:  {auc_pr:.4f}')
    print(f'  AUC-ROC: {auc_roc:.4f}')
    print(f'  F1:      {f1:.4f}')
    print(f'  Best iteration: {lgbm_model.best_iteration_}')


Training until validation scores don't improve for 50 rounds
[100]	valid_0's average_precision: 0.457217
[200]	valid_0's average_precision: 0.486874
[300]	valid_0's average_precision: 0.498719
[400]	valid_0's average_precision: 0.506905
[500]	valid_0's average_precision: 0.512351
[600]	valid_0's average_precision: 0.516321
[700]	valid_0's average_precision: 0.519574
[800]	valid_0's average_precision: 0.523271
[900]	valid_0's average_precision: 0.523982
Early stopping, best iteration is:
[860]	valid_0's average_precision: 0.524382
LightGBM Results:
  AUC-PR:  0.5244
  AUC-ROC: 0.9027
  F1:      0.3882
  Best iteration: 860
🏃 View run lightgbm_baseline at: https://germanywestcentral.api.azureml.ms/mlflow/v2.0/subscriptions/1462cf45-0a1d-4624-a94a-ce9949a6ffd4/resourceGroups/shaliq_study/providers/Microsoft.MachineLearningServices/workspaces/shaliq_demo/#/experiments/51a2010c-aa0f-4cbe-a06b-ae4cdfbda8ca/runs/e8f334fa-3426-45e1-b539-6d3a8b68e2ef
🧪 View experiment at: https://germanywestcen

## Chapter 5 — CatBoost

CatBoost handles categorical features natively — no label encoding needed.
It builds symmetric trees and uses ordered boosting to reduce overfitting.

Key difference:
- `auto_class_weights='Balanced'` — automatic class weight calculation
- `cat_features` — pass categorical column indices directly, no encoding needed
- `eval_metric='PRAUC'` — AUC-PR in CatBoost notation

In [7]:
from catboost import CatBoostClassifier

catboost_params = {
    'iterations': 1000,
    'depth': 6,
    'learning_rate': 0.05,
    'auto_class_weights': 'Balanced',
    'eval_metric': 'PRAUC',
    'early_stopping_rounds': 50,
    'random_seed': 42,
    'verbose': 100
}

with mlflow.start_run(run_name='catboost_baseline'):
    mlflow.log_params(catboost_params)
    
    cat_model = CatBoostClassifier(**catboost_params)
    cat_model.fit(
        X_train, y_train,
        eval_set=(X_val, y_val)
    )
    
    y_pred_proba = cat_model.predict_proba(X_val)[:, 1]
    y_pred       = (y_pred_proba >= 0.5).astype(int)
    
    auc_pr  = average_precision_score(y_val, y_pred_proba)
    auc_roc = roc_auc_score(y_val, y_pred_proba)
    f1      = f1_score(y_val, y_pred)
    
    mlflow.log_metric('auc_pr',  auc_pr)
    mlflow.log_metric('auc_roc', auc_roc)
    mlflow.log_metric('f1',      f1)
    joblib.dump(cat_model, 'models/catboost_baseline.pkl')
    
    print(f'CatBoost Results:')
    print(f'  AUC-PR:  {auc_pr:.4f}')
    print(f'  AUC-ROC: {auc_roc:.4f}')
    print(f'  F1:      {f1:.4f}')

0:	learn: 0.8442860	test: 0.8217038	best: 0.8217038 (0)	total: 200ms	remaining: 3m 19s
100:	learn: 0.9084001	test: 0.8869058	best: 0.8869058 (100)	total: 16.9s	remaining: 2m 29s
200:	learn: 0.9226817	test: 0.8960207	best: 0.8960207 (200)	total: 34.2s	remaining: 2m 15s
300:	learn: 0.9342717	test: 0.9030920	best: 0.9031147 (299)	total: 50.3s	remaining: 1m 56s
400:	learn: 0.9436839	test: 0.9083232	best: 0.9083232 (400)	total: 1m 6s	remaining: 1m 40s
500:	learn: 0.9505844	test: 0.9110838	best: 0.9110838 (500)	total: 1m 23s	remaining: 1m 23s
600:	learn: 0.9558833	test: 0.9133026	best: 0.9133110 (597)	total: 1m 39s	remaining: 1m 6s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.9134262531
bestIteration = 605

Shrink model to first 606 iterations.
CatBoost Results:
  AUC-PR:  0.4983
  AUC-ROC: 0.9059
  F1:      0.3396
🏃 View run catboost_baseline at: https://germanywestcentral.api.azureml.ms/mlflow/v2.0/subscriptions/1462cf45-0a1d-4624-a94a-ce9949a6ffd4/resourceGroups/sha

## Chapter 6 — Model Comparison

Compare all three models side by side and pick the best for threshold optimisation.

In [8]:
from sklearn.metrics import recall_score


results = {}

for name, model in [('XGBoost', xgb_model), ('LightGBM', lgbm_model), ('CatBoost', cat_model)]:
    y_prob = model.predict_proba(X_val)[:, 1]
    results[name] = {
        'AUC-PR':  round(average_precision_score(y_val, y_prob), 4),
        'AUC-ROC': round(roc_auc_score(y_val, y_prob), 4),
        'F1':      round(f1_score(y_val, (y_prob >= 0.5).astype(int)), 4),
        'Recall':  round(recall_score(y_val, y_pred), 4)
    }

results_df = pd.DataFrame(results).T.sort_values('AUC-PR', ascending=False)
print(results_df.sort_values('AUC-PR', ascending=False))

best_model_name = results_df['AUC-PR'].head(1)
print(f'\nBest model: {best_model_name}')

          AUC-PR  AUC-ROC      F1  Recall
LightGBM  0.5244   0.9027  0.3882  0.7231
XGBoost   0.5190   0.8978  0.4103  0.7231
CatBoost  0.4983   0.9059  0.3396  0.7231

Best model: LightGBM    0.5244
Name: AUC-PR, dtype: float64
